# QuantiPhy — geometric vision backendRuns the Grounding-DINO backend over the **159-row validation split** and scores it with our ownharness, which reproduces the organizers' published GPT-5.1 number to within 0.002.**Bar to beat:** GPT-5.1 scores **0.4856** on this split. Human average is 0.556, top humans 0.72.Runtime: **T4 is enough** for Grounding-DINO. Change it under Runtime > Change runtime type.The point of this notebook is one number plus a coverage breakdown — not a submission. Detectionalone is the smallest thing that gives us a measurable score; SAM2 masks and CoTracker trajectoriescome after we know what detection alone buys.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader!pip install -q transformers accelerate opencv-python-headless huggingface_hub

## 1. Get the codePush the `quantiphy` package to GitHub and set `REPO_URL`. Until then, zip the local folder andupload it — the fallback cell handles that.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/prarabdhmisra/quantiphy.git"import os, sys, pathlibif REPO_URL:    !git clone -q $REPO_URL /content/quantiphy-repo    ROOT = "/content/quantiphy-repo"else:    from google.colab import files    print("Upload a zip of the quantiphy/ project folder:")    up = files.upload()    name = next(iter(up))    !unzip -q -o "$name" -d /content/quantiphy-repo    ROOT = "/content/quantiphy-repo"    inner = [p for p in pathlib.Path(ROOT).iterdir() if (p / "quantiphy").is_dir()]    if inner: ROOT = str(inner[0])sys.path.insert(0, ROOT)os.chdir(ROOT)from quantiphy.scoring import score, category_labelsfrom quantiphy.parsing import build_requestfrom quantiphy.solver import solve_rowprint("package loaded from", ROOT)

## 2. Pull validation data and videos (~160 clips, small)

In [ ]:
from huggingface_hub import snapshot_downloadVAL = snapshot_download(repo_id="PaulineLi/QuantiPhy-validation", repo_type="dataset")print(VAL)!ls "$VAL" | head

In [ ]:
import pandas as pd, pathlibval = pd.read_csv(pathlib.Path(VAL) / "validation_dataset.csv", encoding="utf-8-sig")val = val[val["ground_truth_posterior"].notna()].reset_index(drop=True)# Video filenames are not always exact: one has a LEADING SPACE, and the column may omit .mp4.video_dir = pathlib.Path(VAL) / "validation_videos"index = {p.name.strip().lower(): p for p in video_dir.glob("*.mp4")}def find_video(video_id):    key = f"{str(video_id).strip().lower()}.mp4"    return index.get(key) or index.get(str(video_id).strip().lower())val["video_path"] = val["video_id"].map(find_video)print(f"{len(val)} rows, {val.video_path.notna().sum()} with a video file")print("MISSING:", val.loc[val.video_path.isna(), "video_id"].tolist()[:5])

## 3. Build the backend`box_threshold` is the main knob. Too high and objects go undetected (the row falls back to a hardzero unless something else fills it); too low and the box snaps to the wrong object. The extent isa median across frames, so occasional bad frames are already tolerated — bias toward recall.

In [ ]:
from quantiphy.backends.grounding import GroundingDinoBackendbackend = GroundingDinoBackend(    box_threshold=0.25,    text_threshold=0.20,    max_frames=48,    batch_size=8,    cache_path="/content/detections.pkl",   # survives a crash; reruns are near-free)print("device:", backend.device)

## 4. Solve

In [ ]:
from tqdm.auto import tqdmimport numpy as npanswers, diagnostics = [], []for _, r in tqdm(val.iterrows(), total=len(val)):    if r["video_path"] is None:        answers.append(np.nan); diagnostics.append("no video"); continue    req = build_request(r)    a = solve_row(req, backend, str(r["video_path"]), min_confidence=0.0)    answers.append(a.value if a.solved else np.nan)    diagnostics.append(a.method if a.solved else a.reason)backend.save_cache()val["geometric"] = answersval["diagnostic"] = diagnosticssolved = val.geometric.notna().sum()print(f"solved geometrically: {solved}/{len(val)} ({100*solved/len(val):.1f}%)")print(val.diagnostic.value_counts().head(12))

## 5. ScoreTwo numbers matter, and they answer different questions.**Coverage-limited** scores only the rows we solved — it tells us whether the *geometry* is right.**As-submitted** puts a zero on every unsolved row, which is what the leaderboard would actuallygive us today. The gap between them is exactly what the fallback arm has to close.

In [ ]:
sub = val.copy()sub["parsed_value"] = sub["geometric"]solved_only = sub[sub.geometric.notna()]try:    print("coverage-limited (solved rows only):", score(solved_only))except ValueError as e:    print("coverage-limited: not scorable —", e)   # a category with no solved rowssub["parsed_value"] = sub["geometric"].fillna(0.0)print("as-submitted (unsolved = 0):", score(sub))print()print("GPT-5.1 reference on this split: macro 0.4856")

## 6. Where is it wrong?Read this before tuning anything. Under this metric a prediction at or above **1.9x** the truthscores zero outright while a 0.5x undershoot still earns 0.4, so systematic *overshoot* is far moreexpensive than undershoot. Ratios cluster near powers of ten mean a scale bug, not a vision bug.

In [ ]:
import numpy as npd = val[val.geometric.notna()].copy()d["ratio"] = d.geometric.abs() / d.ground_truth_posteriorprint(d.ratio.describe(percentiles=[.1,.25,.5,.75,.9]))print()print(f"over 1.9x (auto-zero): {(d.ratio>=1.9).mean():.1%}")print(f"within 1.1x          : {((d.ratio>0.9)&(d.ratio<1.1)).mean():.1%}")print(f"|log10 ratio| > 1    : {(np.abs(np.log10(d.ratio))>1).mean():.1%}  <- scale bugs")worst = d.assign(err=(np.log10(d.ratio)).abs()).nlargest(12, "err")print(worst[["question","ground_truth_prior","geometric","ground_truth_posterior","ratio"]]      .to_string(max_colwidth=52))

## 7. Next1. **Triage section 6 first.** Order-of-magnitude errors are scale/unit bugs and are cheap to fix;   2x errors are genuine measurement error and are not.2. Add **SAM2 masks** for extent — bounding boxes overestimate non-rectangular objects, and every   overestimate is metric-expensive.3. Add **CoTracker3** for trajectories on the rows where the quadratic fit quality is low.4. Build the **fallback arm** (VLM estimate) and fuse in log space. Until then, unsolved rows are   hard zeros.5. For the full 3,289-row test run, prefer **HF Jobs** over Colab — see the README.